# Original longitude-gradient challenge after HEALPix-to-latitude–longitude remapping

This notebook evaluates ground-truth and decoded HEALPix fields with the exact longitude-derivative calculation from `03-spatial-gradient.ipynb`. Each selected HEALPix map is first interpolated onto the original challenge grid: 721 latitudes ordered from $90^\circ$ to $-90^\circ$ and 1440 longitudes from $0^\circ$ to $359.75^\circ$, both spaced by $0.25^\circ$.

The workflow is: **aligned HEALPix maps → common 0.25° grid → original `xarray.roll` derivative → absolute-error-bound evaluation**. Remapping both fields to the same grid isolates compression error after a shared interpolation step. The interpolation can still smooth both maps relative to their native HEALPix values.

The corrected challenge derivative is preserved literally: roll by -5 supplies the eastern value, roll by +5 supplies the western value, and their wrapped coordinate difference is $2.5^\circ$.

## 1. Environment and imports

Run this notebook in the `climgen` environment. The cell configures writable plotting caches and imports xarray, Healpy, plotting, and progress-bar dependencies. No model or GPU is needed.

In [ ]:
import json
import os
import platform
import re
import warnings
from functools import lru_cache
from pathlib import Path
from urllib.parse import urlsplit

CACHE_ROOT = Path('/tmp') / f'healpix_latlon_gradient_cache_{os.getuid()}'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(CACHE_ROOT / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_ROOT))

import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from tqdm.auto import tqdm

display(pd.Series({
    'Python': platform.python_version(),
    'xarray': xr.__version__,
    'healpy': hp.__version__,
}).to_frame('version'))

## 2. Editable analysis configuration

Provide the original HEALPix Zarr store and the decoded Zarr exported by the inference notebook. `VARIABLES` uses the same conventions as the compression notebooks: timestep ranges are end-exclusive, variables may be grouped under `2D` and `3D`, and `level_indices=None` selects every level. Delete `timesteps` to evaluate every decoded timestep and leave the variable groups absent to use every compatible variable.

Ordering is normally inferred from metadata. Set it explicitly only if the store does not identify NESTED or RING ordering. The target grid is intentionally fixed to the original challenge geometry and is not an editable approximation.

In [ ]:
GROUND_TRUTH_ZARR = os.environ.get(
    'HPX_GRADIENT_GROUND_TRUTH', '/work/bm1235/k202181/ngc4008/ngc4008_P1D_9.zarr', 
    #'HPX_GRADIENT_GROUND_TRUTH', '/p/project1/training2640/meuer1/data/nextGEMS_level9.zarr'
)
DECODED_ZARR = os.environ.get(
    'HPX_GRADIENT_DECODED', '/work/bd1560/k204233/FieldSpaceNN/notebooks/evaluations/hackathon_hyperprior/hus_only_finetune/decoded_fields.zarr'
    #'HPX_GRADIENT_DECODED', '/p/home/jusers/meuer1/jusuf/FieldSpace-compression/notebooks/evaluations/hackathon_hyperprior/hpx9_tas/decoded_fields.zarr'
)
OUTPUT_DIR = Path(os.environ.get(
    'HPX_GRADIENT_OUTPUT', './healpix_spatial_gradient_results'
))

VARIABLES = {
    'timesteps': [0],  # Delete this entry to evaluate all decoded timesteps.
    # With no groups, every common numeric HEALPix variable and all levels are used.
    # '2D': {'tas': {}},
    '2D': {'hus': {'level_indices': [-2]}},
    #'3D': {'ua': {'level_indices': [-36]}},
}

ABSOLUTE_GRADIENT_ERROR_BOUND = 1e-4
GROUND_TRUTH_ORDERING = 'auto'  # 'auto', 'NESTED', or 'RING'
DECODED_ORDERING = 'auto'       # 'auto', 'NESTED', or 'RING'

## 3. Selection and alignment helpers

These helpers open local or remote Zarr stores, identify HEALPix/time/level dimensions, expand requested index ranges, and align reconstructed samples with the corresponding ground truth. If the decoded export contains `source_time_index`, it is used as the authoritative mapping. Incompatible grids or coordinates fail before any full map is loaded.

In [ ]:
def is_remote(location):
    scheme = urlsplit(str(location)).scheme.lower()
    return bool(scheme and scheme != 'file')


def open_zarr_store(location):
    if not is_remote(location) and not Path(location).expanduser().exists():
        raise FileNotFoundError(f'Zarr store does not exist: {location}')
    return xr.open_zarr(str(location), consolidated=None)


def expand_positions(selection, size, label):
    if selection is None:
        return list(range(size))
    items = selection if isinstance(selection, (list, tuple)) else [selection]
    positions = []
    for item in items:
        if isinstance(item, (int, np.integer)) and not isinstance(item, bool):
            positions.append(int(item))
        elif isinstance(item, str) and (match := re.fullmatch(r'\s*(\d+)\s*-\s*(\d+)\s*', item)):
            start, stop = map(int, match.groups())
            if stop <= start:
                raise ValueError(f'Invalid {label} range {item!r}: stop must exceed start.')
            positions.extend(range(start, stop))
        else:
            raise ValueError(f'Invalid {label} entry {item!r}.')
    if not positions or min(positions) < 0 or max(positions) >= size:
        raise IndexError(f'{label} positions must fall in [0, {size - 1}].')
    if len(positions) != len(set(positions)):
        raise ValueError(f'{label} selection contains duplicates or overlapping ranges.')
    return positions


def level_positions(selection, size, variable):
    if selection is None:
        return list(range(size))
    items = [selection] if isinstance(selection, (int, np.integer)) else list(selection)
    if not items or any(not isinstance(index, (int, np.integer)) for index in items):
        raise ValueError(f'level_indices for {variable!r} must contain integers.')
    positions = [int(index) % size if int(index) < 0 else int(index) for index in items]
    if min(positions) < 0 or max(positions) >= size:
        raise IndexError(f'level_indices for {variable!r} exceed the decoded depth {size}.')
    if len(positions) != len(set(positions)):
        raise ValueError(f'level_indices for {variable!r} contains duplicates.')
    return positions


def flatten_variable_configuration(configuration):
    entries = {str(key): value for key, value in configuration.items() if str(key) != 'timesteps'}
    if any(group in entries for group in ('2D', '3D')):
        unexpected = set(entries) - {'2D', '3D'}
        if unexpected:
            raise ValueError(f'Unexpected entries beside 2D/3D groups: {sorted(unexpected)}')
        flattened = {}
        for group in ('2D', '3D'):
            for variable, specification in (entries.get(group) or {}).items():
                if variable in flattened:
                    raise ValueError(f'Variable {variable!r} occurs in more than one group.')
                flattened[str(variable)] = specification or {}
        return flattened
    return {str(variable): specification or {} for variable, specification in entries.items()}


def healpix_dimension(data_array):
    candidates = []
    for dimension, size in data_array.sizes.items():
        try:
            nside = hp.npix2nside(int(size))
        except ValueError:
            continue
        if nside > 0 and nside & (nside - 1) == 0:
            priority = 0 if ('cell' in dimension.lower() or 'pixel' in dimension.lower()) else 1
            candidates.append((priority, dimension, nside))
    if not candidates:
        raise ValueError(f'{data_array.name!r} has no valid HEALPix cell dimension.')
    candidates.sort()
    if len(candidates) > 1 and candidates[0][0] == candidates[1][0]:
        raise ValueError(f'Ambiguous HEALPix dimensions for {data_array.name!r}: {candidates}.')
    return candidates[0][1], candidates[0][2]


def axis_dimension(data_array, kind, excluded):
    aliases = {
        'time': {'time', 'times', 'date', 'datetime'},
        'level': {'lev', 'level', 'levels', 'plev', 'height', 'depth', 'altitude', 'model_level'},
    }[kind]
    axis = {'time': 'T', 'level': 'Z'}[kind]
    candidates = []
    for dimension in data_array.dims:
        if dimension in excluded:
            continue
        coordinate = data_array.coords.get(dimension)
        attrs = {} if coordinate is None else {str(k).lower(): str(v).lower() for k, v in coordinate.attrs.items()}
        score = 50 * (dimension.lower() in aliases) + 100 * (attrs.get('axis', '').upper() == axis)
        if kind == 'time':
            score += 100 * (attrs.get('standard_name') == 'time')
        if score:
            candidates.append((score, dimension))
    return max(candidates)[1] if candidates else None


def detect_ordering(dataset, requested, label):
    requested = str(requested).upper()
    if requested in {'NESTED', 'RING'}:
        return requested
    if requested != 'AUTO':
        raise ValueError(f'{label}_ORDERING must be auto, NESTED, or RING.')
    for attributes in [dataset.attrs] + [value.attrs for value in dataset.variables.values()]:
        for key, value in attributes.items():
            key_lower, value_lower = str(key).lower(), str(value).lower()
            if 'order' in key_lower and 'nest' in value_lower:
                return 'NESTED'
            if 'order' in key_lower and 'ring' in value_lower:
                return 'RING'
            if 'nest' in key_lower and isinstance(value, (bool, np.bool_)):
                return 'NESTED' if bool(value) else 'RING'
    warnings.warn(f'No HEALPix ordering metadata found for {label}; assuming NESTED.', stacklevel=2)
    return 'NESTED'


def coordinate_positions(reference, requested, label):
    positions = reference.to_index().get_indexer(pd.Index(np.asarray(requested)))
    if (positions < 0).any():
        missing = np.asarray(requested)[positions < 0]
        raise ValueError(f'{label} coordinates are absent from ground truth: {missing.tolist()}')
    return positions.tolist()

## 4. Prepare aligned HEALPix maps

The next helpers apply timestep and level selections without changing the original stores. They yield one aligned ground-truth/reconstruction pair at a time, keeping full-evaluation memory bounded. A variable must reduce to one HEALPix map after selecting one time and one level.

In [ ]:
def prepare_variable_pair(ground_truth, decoded, variable, specification, timestep_selection):
    if variable not in ground_truth.data_vars or variable not in decoded.data_vars:
        raise KeyError(f'{variable!r} must be a data variable in both stores.')
    truth, reconstruction = ground_truth[variable], decoded[variable]
    truth_cell, truth_nside = healpix_dimension(truth)
    decoded_cell, decoded_nside = healpix_dimension(reconstruction)
    if truth_nside != decoded_nside:
        raise ValueError(f'{variable!r} has NSIDE {truth_nside} in ground truth and {decoded_nside} decoded.')

    decoded_time = axis_dimension(reconstruction, 'time', {decoded_cell})
    truth_time = axis_dimension(truth, 'time', {truth_cell})
    if decoded_time is None:
        if timestep_selection not in (None, [], [0]):
            raise ValueError(f'{variable!r} has no decoded time dimension.')
        selected_times = [0]
    else:
        selected_times = expand_positions(timestep_selection, reconstruction.sizes[decoded_time], 'timestep')
        reconstruction = reconstruction.isel({decoded_time: selected_times})
        if truth_time is None:
            raise ValueError(f'{variable!r} has decoded time but no ground-truth time dimension.')
        if 'source_time_index' in decoded and decoded_time in decoded['source_time_index'].dims:
            source_positions = np.asarray(
                decoded['source_time_index'].isel({decoded_time: selected_times}).values, dtype=int
            )
            if source_positions.min() < 0 or source_positions.max() >= truth.sizes[truth_time]:
                raise IndexError('Decoded source_time_index falls outside the ground-truth store.')
            truth = truth.isel({truth_time: source_positions.tolist()})
        elif decoded_time in reconstruction.coords and truth_time in truth.coords:
            truth = truth.isel({truth_time: coordinate_positions(
                truth[truth_time], reconstruction[decoded_time].values, 'time'
            )})
        elif truth.sizes[truth_time] == decoded.sizes[decoded_time]:
            truth = truth.isel({truth_time: selected_times})
        else:
            raise ValueError('Cannot align decoded and ground-truth time axes.')

    decoded_level = axis_dimension(reconstruction, 'level', {decoded_cell, decoded_time})
    truth_level = axis_dimension(truth, 'level', {truth_cell, truth_time})
    if decoded_level is None:
        if specification.get('level_indices') is not None:
            raise ValueError(f'{variable!r} has no decoded level dimension.')
        selected_levels = [0]
        if truth_level is not None:
            raise ValueError(f'{variable!r} has a ground-truth level dimension but decoded data does not.')
    else:
        selected_levels = level_positions(
            specification.get('level_indices'), reconstruction.sizes[decoded_level], variable
        )
        reconstruction = reconstruction.isel({decoded_level: selected_levels})
        if truth_level is None:
            raise ValueError(f'{variable!r} has decoded levels but no ground-truth level dimension.')
        if decoded_level in reconstruction.coords and truth_level in truth.coords:
            truth = truth.isel({truth_level: coordinate_positions(
                truth[truth_level], reconstruction[decoded_level].values, 'level'
            )})
        elif truth.sizes[truth_level] == decoded.sizes[decoded_level]:
            truth = truth.isel({truth_level: selected_levels})
        else:
            raise ValueError('Cannot align decoded and ground-truth level axes.')

    if set(truth.dims) - {truth_cell, truth_time, truth_level, None}:
        raise ValueError(f'{variable!r} has unsupported ground-truth dimensions.')
    if set(reconstruction.dims) - {decoded_cell, decoded_time, decoded_level, None}:
        raise ValueError(f'{variable!r} has unsupported decoded dimensions.')
    return {
        'truth': truth, 'decoded': reconstruction, 'truth_time': truth_time,
        'decoded_time': decoded_time, 'truth_level': truth_level,
        'decoded_level': decoded_level, 'time_positions': selected_times,
        'level_positions': selected_levels, 'nside': truth_nside,
    }


def iter_aligned_maps(pair):
    n_times = pair['decoded'].sizes[pair['decoded_time']] if pair['decoded_time'] else 1
    n_levels = pair['decoded'].sizes[pair['decoded_level']] if pair['decoded_level'] else 1
    for time_offset in range(n_times):
        for level_offset in range(n_levels):
            truth_indexers, decoded_indexers = {}, {}
            if pair['truth_time']:
                truth_indexers[pair['truth_time']] = time_offset
                decoded_indexers[pair['decoded_time']] = time_offset
            if pair['truth_level']:
                truth_indexers[pair['truth_level']] = level_offset
                decoded_indexers[pair['decoded_level']] = level_offset
            truth_map = np.asarray(pair['truth'].isel(truth_indexers).values).squeeze()
            decoded_map = np.asarray(pair['decoded'].isel(decoded_indexers).values).squeeze()
            if truth_map.ndim != 1 or decoded_map.ndim != 1:
                raise ValueError(f'Expected one-dimensional HEALPix maps, got {truth_map.shape} and {decoded_map.shape}.')
            yield {
                'truth': truth_map, 'decoded': decoded_map,
                'timestep_position': pair['time_positions'][time_offset],
                'level_position': pair['level_positions'][level_offset],
                'time_value': (pair['decoded'][pair['decoded_time']].values[time_offset]
                               if pair['decoded_time'] and pair['decoded_time'] in pair['decoded'].coords else None),
                'level_value': (pair['decoded'][pair['decoded_level']].values[level_offset]
                                if pair['decoded_level'] and pair['decoded_level'] in pair['decoded'].coords else None),
            }

## 5. Remap to the original grid and apply the original derivative

The target coordinates are fixed to the original 0.25° global grid. `healpix_to_original_grid` uses Healpy's bilinear interpolation at every target cell center. `differentiate_along_longitude` is copied directly from the corrected challenge notebook. Non-finite input is rejected because Healpy interpolation across missing pixels would be ambiguous.

In [ ]:
TARGET_LATITUDES = np.linspace(90.0, -90.0, 721, dtype=np.float64)
TARGET_LONGITUDES = np.arange(1440, dtype=np.float64) * 0.25


@lru_cache(maxsize=1)
def target_healpy_angles():
    latitude, longitude = np.meshgrid(TARGET_LATITUDES, TARGET_LONGITUDES, indexing='ij')
    theta = np.deg2rad(90.0 - latitude).ravel()
    phi = np.deg2rad(longitude).ravel()
    return theta, phi


def healpix_to_original_grid(values, nside, ordering, name, attrs):
    values = np.asarray(values, dtype=np.float64).squeeze()
    expected = hp.nside2npix(nside)
    if values.ndim != 1 or values.size != expected:
        raise ValueError(f'Expected one map with {expected} pixels, got {values.shape}.')
    if not np.isfinite(values).all():
        raise ValueError(f'{name!r} contains {int((~np.isfinite(values)).sum())} non-finite values.')
    ring_values = hp.reorder(values, n2r=True) if ordering == 'NESTED' else values
    theta, phi = target_healpy_angles()
    remapped = hp.get_interp_val(ring_values, theta, phi, nest=False).reshape(721, 1440)
    result = xr.DataArray(
        remapped, dims=('lat', 'lon'),
        coords={'lat': TARGET_LATITUDES, 'lon': TARGET_LONGITUDES},
        name=name, attrs=dict(attrs),
    )
    result.attrs.setdefault('long_name', name)
    return result


def differentiate_along_longitude(da: xr.DataArray) -> xr.DataArray:
    # Exact implementation from 03-spatial-gradient.ipynb.
    # dX / dlon = (x[i+5] - x[i-5]) / ((lon[i+5] - lon[i-5]) % 360)
    da_dXdLon = (da.roll(lon=5) - da.roll(lon=-5)) / (
        np.mod(da.lon.roll(lon=5) - da.lon.roll(lon=-5), 360)
    )
    da_dXdLon.attrs.update(
        long_name=f"{da.long_name} derivative along longitude",
        units=(f"{da.units} degree**-1" if hasattr(da, "units") else "degree**-1"),
    )
    return da_dXdLon


def remapped_derivative_comparison(
    item, pair, variable, truth_order, decoded_order, truth_attrs, decoded_attrs
):
    truth_latlon = healpix_to_original_grid(
        item['truth'], pair['nside'], truth_order, variable, truth_attrs
    )
    decoded_latlon = healpix_to_original_grid(
        item['decoded'], pair['nside'], decoded_order, variable, decoded_attrs
    )
    truth_derivative = differentiate_along_longitude(truth_latlon)
    decoded_derivative = differentiate_along_longitude(decoded_latlon)
    return (
        truth_latlon, decoded_latlon, truth_derivative, decoded_derivative,
        decoded_derivative - truth_derivative,
    )

## 6. Open and validate both stores

Only metadata is inspected here. The cell selects compatible variables, validates their HEALPix resolutions and requested indices, detects ordering, and reports the number of full 721×1440 maps that will be produced temporarily during evaluation. Confirm any warning that automatic ordering fell back to NESTED.

In [ ]:
if not np.isfinite(ABSOLUTE_GRADIENT_ERROR_BOUND) or ABSOLUTE_GRADIENT_ERROR_BOUND < 0:
    raise ValueError('ABSOLUTE_GRADIENT_ERROR_BOUND must be finite and non-negative.')
if TARGET_LATITUDES.shape != (721,) or TARGET_LONGITUDES.shape != (1440,):
    raise RuntimeError('The target grid must remain 721 × 1440.')
if not np.allclose(np.diff(TARGET_LATITUDES), -0.25) or not np.allclose(np.diff(TARGET_LONGITUDES), 0.25):
    raise RuntimeError('The target latitude and longitude spacing must remain 0.25 degrees.')

ground_truth = open_zarr_store(GROUND_TRUTH_ZARR)
decoded = open_zarr_store(DECODED_ZARR)
truth_ordering = detect_ordering(ground_truth, GROUND_TRUTH_ORDERING, 'GROUND_TRUTH')
decoded_ordering = detect_ordering(decoded, DECODED_ORDERING, 'DECODED')
variable_specs = flatten_variable_configuration(VARIABLES)

if not variable_specs:
    common = []
    for name in sorted(set(ground_truth.data_vars) & set(decoded.data_vars)):
        try:
            _, truth_nside = healpix_dimension(ground_truth[name])
            _, decoded_nside = healpix_dimension(decoded[name])
        except ValueError:
            continue
        if truth_nside == decoded_nside and np.issubdtype(ground_truth[name].dtype, np.number):
            common.append(name)
    variable_specs = {name: {} for name in common}
if not variable_specs:
    raise ValueError('The stores have no common numeric HEALPix variables.')

geometry_rows = []
for variable in variable_specs:
    pair = prepare_variable_pair(
        ground_truth, decoded, variable, variable_specs[variable], VARIABLES.get('timesteps')
    )
    geometry_rows.append({
        'variable': variable, 'HPX level': int(np.log2(pair['nside'])), 'NSIDE': pair['nside'],
        'selected timesteps': len(pair['time_positions']), 'selected levels': len(pair['level_positions']),
        'maps': len(pair['time_positions']) * len(pair['level_positions']),
    })
geometry = pd.DataFrame(geometry_rows).set_index('variable')
display(pd.Series({
    'Ground truth': str(GROUND_TRUTH_ZARR), 'Decoded reconstruction': str(DECODED_ZARR),
    'Ground-truth ordering': truth_ordering, 'Decoded ordering': decoded_ordering,
    'Target grid': '721 lat × 1440 lon (0.25°)',
    'Derivative expression': 'original xarray.roll expression',
    'Wrapped denominator': '2.5°',
    'Absolute error bound': f'{ABSOLUTE_GRADIENT_ERROR_BOUND:g} per degree',
    'Total maps': int(geometry['maps'].sum()),
}).to_frame('value'))
display(geometry)

## 7. Remap and evaluate

Each selected time/level pair is independently remapped and differentiated. A target-grid cell violates the challenge when the absolute decoded-minus-original derivative error exceeds the configured bound. Only scalar metrics are retained, so the next pair can reuse the memory. The table includes actual derivative ranges to make direct comparison with the source challenge easier.

In [ ]:
metric_rows = []
for variable in tqdm(list(variable_specs), desc='Variables'):
    pair = prepare_variable_pair(
        ground_truth, decoded, variable, variable_specs[variable], VARIABLES.get('timesteps')
    )
    map_count = len(pair['time_positions']) * len(pair['level_positions'])
    truth_attrs = dict(ground_truth[variable].attrs)
    decoded_attrs = dict(decoded[variable].attrs)
    for item in tqdm(iter_aligned_maps(pair), total=map_count, desc=f'{variable} maps', leave=False):
        _, _, truth_derivative, decoded_derivative, signed_error = remapped_derivative_comparison(
            item, pair, variable, truth_ordering, decoded_ordering,
            truth_attrs, decoded_attrs,
        )
        error_values = np.asarray(signed_error.values)
        truth_values = np.asarray(truth_derivative.values)
        decoded_values = np.asarray(decoded_derivative.values)
        absolute_error = np.abs(error_values)
        violations = absolute_error > ABSOLUTE_GRADIENT_ERROR_BOUND
        metric_rows.append({
            'variable': variable, 'timestep_position': item['timestep_position'],
            'time_value': str(item['time_value']), 'level_position': item['level_position'],
            'level_value': str(item['level_value']), 'grid_cell_count': error_values.size,
            'violation_count': int(violations.sum()), 'violation_fraction': float(violations.mean()),
            'mean_absolute_derivative_error': float(absolute_error.mean()),
            'derivative_error_rmse': float(np.sqrt(np.mean(error_values ** 2))),
            'max_absolute_derivative_error': float(absolute_error.max()),
            'truth_derivative_min': float(truth_values.min()),
            'truth_derivative_max': float(truth_values.max()),
            'decoded_derivative_min': float(decoded_values.min()),
            'decoded_derivative_max': float(decoded_values.max()),
        })

metrics = pd.DataFrame(metric_rows)
if metrics.empty:
    raise RuntimeError('No maps were evaluated.')
total_cells = int(metrics['grid_cell_count'].sum())
total_violations = int(metrics['violation_count'].sum())
summary = pd.Series({
    'Result': 'PASS' if total_violations == 0 else 'FAIL',
    'Evaluated maps': len(metrics), 'Evaluated target-grid cells': total_cells,
    'Violating cells': total_violations, 'Violation fraction': total_violations / total_cells,
    'Worst map maximum error': metrics['max_absolute_derivative_error'].max(),
    'Absolute error bound': ABSOLUTE_GRADIENT_ERROR_BOUND,
})
display(summary.to_frame('value'))
display(metrics)

## 8. Plot one remapped comparison

These controls affect only this figure. The selected pair is remapped again so the full evaluation does not retain large arrays. The plotting logic follows the challenge helper: a 15×4 three-map layout, an equirectangular longitude–latitude view, 22 linearly spaced color levels, five colorbar ticks, and **independent actual minimum/maximum limits** for the original and decoded derivatives. The signed error panel uses the fixed $\pm$ error bound and reports violations with the same formatting as the source notebook. Standard Matplotlib axes are used deliberately so rendering does not depend on Cartopy or a Natural Earth download.

The displayed range table separates field ranges from derivative ranges. Matching field extrema does not imply matching derivatives: the earlier regular-grid → HEALPix interpolation and this notebook's HEALPix → regular-grid interpolation change neighboring values and are not inverse operations.

In [ ]:
PLOT_VARIABLE = next(iter(variable_specs))
PLOT_TIMESTEP_INDEX = 0
PLOT_LEVEL_INDEX = 0

if PLOT_VARIABLE not in variable_specs:
    raise ValueError(f'PLOT_VARIABLE must be one of {list(variable_specs)}.')
plot_specification = dict(variable_specs[PLOT_VARIABLE])
plot_cell, _ = healpix_dimension(decoded[PLOT_VARIABLE])
plot_time = axis_dimension(decoded[PLOT_VARIABLE], 'time', {plot_cell})
plot_level = axis_dimension(decoded[PLOT_VARIABLE], 'level', {plot_cell, plot_time})
if plot_level is not None:
    plot_specification['level_indices'] = [PLOT_LEVEL_INDEX]
elif PLOT_LEVEL_INDEX != 0:
    raise IndexError(f'{PLOT_VARIABLE!r} is 2-D; PLOT_LEVEL_INDEX must be 0.')
plot_pair = prepare_variable_pair(
    ground_truth, decoded, PLOT_VARIABLE, plot_specification, [PLOT_TIMESTEP_INDEX]
)
plot_item = next(iter_aligned_maps(plot_pair))
truth_latlon, decoded_latlon, truth_derivative, decoded_derivative, signed_error = (
    remapped_derivative_comparison(
        plot_item, plot_pair, PLOT_VARIABLE, truth_ordering, decoded_ordering,
        dict(ground_truth[PLOT_VARIABLE].attrs),
        dict(decoded[PLOT_VARIABLE].attrs),
    )
)

# Show the numerical ranges before plotting; these are the values used below.
range_table = pd.DataFrame([
    {'quantity': 'Remapped original field', 'minimum': float(truth_latlon.min()),
     'maximum': float(truth_latlon.max())},
    {'quantity': 'Remapped decoded field', 'minimum': float(decoded_latlon.min()),
     'maximum': float(decoded_latlon.max())},
    {'quantity': 'Original longitude derivative', 'minimum': float(truth_derivative.min()),
     'maximum': float(truth_derivative.max())},
    {'quantity': 'Decoded longitude derivative', 'minimum': float(decoded_derivative.min()),
     'maximum': float(decoded_derivative.max())},
    {'quantity': 'Decoded − original derivative', 'minimum': float(signed_error.min()),
     'maximum': float(signed_error.max())},
]).set_index('quantity')
display(range_table)

# Reproduce the challenge's violation percentage formatting exactly.
violation_fraction = np.mean(
    ~(np.abs(decoded_derivative - truth_derivative) <= ABSOLUTE_GRADIENT_ERROR_BOUND)
)
violation_label = (
    0 if violation_fraction == 0
    else np.format_float_positional(100 * violation_fraction, precision=1, min_digits=1) + '%'
)
if violation_label == '0.0%':
    violation_label = '<0.05%'

def challenge_quickplot(data_array, axis, title, vrange=None, error=False):
    """Match the source ranges and colors on backend-independent lon/lat axes."""
    data_min = float(np.nanmin(data_array.values))
    data_max = float(np.nanmax(data_array.values))
    if vrange is None:
        lower, upper = data_min, data_max
        if error:
            upper = max(abs(lower), abs(upper))
            lower = -upper
    else:
        lower, upper = map(float, vrange)
    if lower == upper:
        upper = lower + np.finfo(float).eps

    levels = np.linspace(lower, upper, 22)
    colorbar_ticks = np.linspace(lower, upper, 5)
    extend = (
        'both' if data_min < lower and data_max > upper
        else 'min' if data_min < lower
        else 'max' if data_max > upper
        else 'neither'
    )
    # The challenge schema uses plasma by default; quickplot overrides errors to coolwarm.
    cmap = plt.get_cmap('coolwarm' if error else 'plasma')
    norm = BoundaryNorm(levels, cmap.N, clip=False)
    # Centre 0–360° longitudes on the Pacific seam for a conventional world map.
    display_array = data_array.assign_coords(
        lon=((data_array.lon + 180) % 360) - 180
    ).sortby('lon')
    axis.set_facecolor('lavenderblush' if error else 'lightgrey')
    image = axis.pcolormesh(
        display_array.lon, display_array.lat, display_array.values, shading='auto',
        cmap=cmap, norm=norm, rasterized=True,
    )
    axis.set_xlim(-180, 180)
    axis.set_ylim(-90, 90)
    axis.set_xticks([-180, -120, -60, 0, 60, 120, 180])
    axis.set_xticklabels(['180°', '120°W', '60°W', '0°', '60°E', '120°E', '180°'])
    axis.set_yticks([-60, -30, 0, 30, 60])
    axis.set_yticklabels(['60°S', '30°S', '0°', '30°N', '60°N'])
    axis.grid(linewidth=0.5, color='#bbbbbb', alpha=0.8)
    axis.tick_params(labelsize=8)
    axis.set_title(title)
    colorbar = figure.colorbar(
        image, ax=axis, orientation='horizontal', pad=0.12,
        ticks=colorbar_ticks, extend=extend,
    )
    colorbar.set_label(data_array.attrs.get('units', ''))

# Match the original Figure(figsize=(15, 4), rows=1, columns=3) layout.
figure, axes = plt.subplots(1, 3, figsize=(15, 4))
original_title = truth_derivative.attrs.get('long_name', 'longitude derivative')
challenge_quickplot(
    truth_derivative, axes[0], title=f'Original {original_title}'
)
challenge_quickplot(
    decoded_derivative, axes[1], title=f'Compressed with {violation_label} violations'
)
error_map = signed_error.assign_attrs(
    long_name=f'{original_title} error'
)
challenge_quickplot(
    error_map, axes[2], error=True,
    vrange=(-ABSOLUTE_GRADIENT_ERROR_BOUND, ABSOLUTE_GRADIENT_ERROR_BOUND),
    title='Absolute Derivative Error over Compressed Data',
)
figure.subplots_adjust(wspace=0.16, bottom=0.18)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_RANGES_PATH = OUTPUT_DIR / f'{PLOT_VARIABLE}_latlon_spatial_gradient_ranges.csv'
FIGURE_PATH = OUTPUT_DIR / f'{PLOT_VARIABLE}_latlon_spatial_gradient.png'
range_table.to_csv(PLOT_RANGES_PATH)
figure.savefig(FIGURE_PATH, dpi=180, bbox_inches='tight')
plt.show()
print(f'Figure: {FIGURE_PATH.resolve()}')

## 9. Save reusable results

This cell stores per-map metrics and a compact JSON summary. The large remapped fields are not written because they are intermediate evaluation products; both input Zarr stores remain unchanged.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = OUTPUT_DIR / 'latlon_spatial_gradient_metrics.csv'
SUMMARY_PATH = OUTPUT_DIR / 'latlon_spatial_gradient_summary.json'
metrics.to_csv(METRICS_PATH, index=False)
summary_payload = {
    'ground_truth_zarr': str(GROUND_TRUTH_ZARR), 'decoded_zarr': str(DECODED_ZARR),
    'variables': list(variable_specs), 'ground_truth_ordering': truth_ordering,
    'decoded_ordering': decoded_ordering, 'target_latitudes': 721, 'target_longitudes': 1440,
    'target_spacing_degrees': 0.25, 'derivative': 'corrected challenge xarray.roll expression',
    'wrapped_denominator_degrees': 2.5,
    'absolute_gradient_error_bound': float(ABSOLUTE_GRADIENT_ERROR_BOUND),
    'result': summary['Result'], 'evaluated_maps': int(summary['Evaluated maps']),
    'evaluated_target_grid_cells': total_cells, 'violating_cells': total_violations,
    'violation_fraction': float(total_violations / total_cells),
    'maximum_absolute_derivative_error': float(metrics['max_absolute_derivative_error'].max()),
}
SUMMARY_PATH.write_text(json.dumps(summary_payload, indent=2) + '\n', encoding='utf-8')
display(pd.Series({
    'Metrics CSV': str(METRICS_PATH.resolve()),
    'Summary JSON': str(SUMMARY_PATH.resolve()),
    'Comparison figure': str(FIGURE_PATH.resolve()),
    'Plotted ranges CSV': str(PLOT_RANGES_PATH.resolve()),
}).to_frame('path'))
ground_truth.close()
decoded.close()

## Interpretation

A `PASS` means every evaluated cell on the original 721×1440 target grid satisfies the challenge's absolute longitude-derivative error bound. Unlike native HEALPix metrics, these cells are not equal area; the reported violation fraction intentionally follows the original notebook and weights every latitude–longitude grid cell equally. Because both inputs undergo the same interpolation, this evaluates reconstruction differences after remapping rather than native-grid differences.